# Public slideshow copy — create & refresh

Creates (or refreshes) a public, read-only Feature Service containing only the project and task fields the analytics-tracker **slideshow** actually reads. Shared with Everyone so lobby-display TVs can show live numbers without an AGO login.

**Idempotent.** First run creates the service and its tables, then loads data. Every subsequent run truncates and reloads. Safe to schedule as a recurring AGO Notebook task.

**Source of truth:** v2 services (`datateam_portfolio_v2`) — official as of 2026-05-17.

**Why not AGO views:** views lock the source schema, blocking edits/migrations on v2. This notebook is the replacement strategy.

**Editing the field list:** add the new field to `PROJECT_FIELDS` or `TASK_FIELDS` below — that's it. The schema-ensure step detects when an existing table is missing one of the expected fields and auto-deletes-then-recreates with the wider schema. (Removing a field works the same way: drop it from the list, next run rebuilds without it.)

**After first run:** copy the printed URLs into `src/agol.js` → `ARCGIS_CONFIG.publicProjectsUrl` / `publicTasksUrl`, bump `APP_VERSION` and the `?v=` cache-bust.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, FeatureLayerCollection

# 'home' resolves to the notebook owner when run inside AGO (manually or scheduled).
gis = GIS("home")
print(f"Authenticated as: {gis.users.me.username} @ {gis.url}")

## Config

Source URLs come from `src/agol.js` → `ARCGIS_CONFIG`. Field lists derived from `src/tabs/overview.js` → `_buildOverviewSlides()`.

In [ ]:
SOURCE_PROJECTS_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/0"
SOURCE_TASKS_URL    = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"

PUBLIC_SERVICE_NAME = "datateam_portfolio_public"

# Slideshow-only fields. Keep narrow on purpose: anything not listed here
# is intentionally excluded from the public copy.
PROJECT_FIELDS = [
    "project_number",       # primary key (JS aliases to .id)
    "title",                # deadline list, debugging
    "status",               # filtered on by nearly every slide
    "category",             # 'Projects by category' slide
    "end_date",             # fallback deadline (JS alias .end)
    "working_due",          # primary deadline
    "actual_end",           # completions: throughput, intake, overdue trend
    "owning_unit",          # new org-cleanup field (replaces itd_team)
    "owning_team",          # new org-cleanup field (replaces data_program_team)
    "data_program_team",    # 'Data Program at a glance' slide — remove once slideshow reads owning_team instead
    "dp_goal",              # legacy derivation of is_data_program
]

TASK_FIELDS = [
    "task_number",          # primary key (JS alias .idx)
    "project_number",       # FK to projects (JS alias .project_id)
    "title",                # deadline list
    "status",               # filters everywhere
    "priority",             # 'Open task priority breakdown' slide
    "start_date",           # intake balance (JS alias .start)
    "due_date",             # fallback deadline (JS alias .due)
    "working_due",          # primary deadline
    "actual_end",           # completions: throughput, intake
]

# Editor-tracking and admin metadata that should never be copied even if
# someone accidentally adds them to PROJECT_FIELDS / TASK_FIELDS above.
ALWAYS_DROP = {"CreationDate", "Creator", "EditDate", "Editor", "GlobalID"}

## Phase 1 — Open source layers

Confirms auth works and prints row counts so you can sanity-check the public copy at the end.

In [ ]:
src_projects = FeatureLayer(SOURCE_PROJECTS_URL, gis=gis)
src_tasks    = FeatureLayer(SOURCE_TASKS_URL,    gis=gis)

src_projects_count = src_projects.query(where="1=1", return_count_only=True)
src_tasks_count    = src_tasks.query(where="1=1", return_count_only=True)
print(f"Source projects ({src_projects.properties.name}): {src_projects_count} rows")
print(f"Source tasks    ({src_tasks.properties.name}): {src_tasks_count} rows")

## Phase 2 — Ensure the public service exists

Looks for an existing Feature Service owned by the running user with the configured name. Creates an empty one if missing.

In [ ]:
# Known item ID for the existing public service. Set this if the search-based
# lookups below can't find it (happens when AGO's content index is stale or
# the item was recently recreated). Leave as None on first-ever run.
KNOWN_PUBLIC_ITEM_ID = "350c8fd92ed944719a903e236ef07658"

def get_or_create_public_service(name: str):
    # 0. Direct lookup by known item ID (most reliable).
    if KNOWN_PUBLIC_ITEM_ID:
        item = gis.content.get(KNOWN_PUBLIC_ITEM_ID)
        if item and item.type == "Feature Service":
            print(f"Public service found by ID: {item.id} ({item.title}, owner={item.owner})")
            return item

    # 1. Exact title match owned by current user.
    me = gis.users.me.username
    hits = [
        i for i in gis.content.search(f'title:"{name}" owner:{me}', item_type="Feature Service")
        if i.title == name
    ]
    if hits:
        print(f"Public service exists (owned by me): {hits[0].id} ({hits[0].title})")
        return hits[0]

    # 2. Fallback: search the org by name and match via URL substring.
    target_url_substr = f"/{name}/FeatureServer"
    for i in gis.content.search(name, item_type="Feature Service", max_items=50):
        if target_url_substr in (i.url or ""):
            print(f"Public service exists (found by URL): {i.id} ({i.title}, owner={i.owner})")
            return i

    # 3. Truly missing — create it.
    print(f"Creating new public service: {name}")
    item = gis.content.create_service(
        name=name,
        service_description="Read-only public copy of selected portfolio fields for the analytics-tracker lobby-display slideshow. Refreshed by scheduled notebook.",
        has_static_data=False,
        max_record_count=4000,
        capabilities="Query",
        service_type="featureService",
    )
    print(f"Created: {item.id}")
    return item

public_item = get_or_create_public_service(PUBLIC_SERVICE_NAME)
public_flc  = FeatureLayerCollection.fromitem(public_item)

## Phase 3 — Ensure each table exists with the filtered schema

For each source layer, build a table definition containing only the kept fields and add it to the public service if missing. We publish as **tables** (no geometry) since the slideshow doesn't render anything spatial.

In [ ]:
def build_table_def(src_layer, keep_fields, new_name):
    src_props = dict(src_layer.properties)
    src_fields = src_props.get("fields", [])
    kept = []
    for f in src_fields:
        nm = f["name"]
        if nm in ALWAYS_DROP:
            continue
        if nm == "ObjectId" or nm in keep_fields:
            kept.append(dict(f))
    return {
        "name": new_name,
        "type": "Table",
        "displayField": src_props.get("displayField") or kept[0]["name"],
        "description": f"Public slideshow copy of {src_props.get('name')} — slideshow-only fields.",
        "objectIdField": "ObjectId",
        "fields": kept,
        "capabilities": "Query",
        "supportsAdvancedQueries": True,
        "hasAttachments": False,
    }

def ensure_table(item_id: str, src_layer, keep_fields, new_name):
    flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
    existing = next((t for t in flc.tables if t.properties.name == new_name), None)

    # If the table exists but is missing any of the expected fields (e.g.
    # PROJECT_FIELDS was widened after the table was first created), drop it
    # so we can recreate with the wider schema. This is the automation the
    # README at the top of the notebook said wasn't there.
    if existing:
        existing_field_names = {f["name"] for f in existing.properties.fields}
        expected_field_names = set(keep_fields) | {"ObjectId"}
        missing = expected_field_names - existing_field_names
        if missing:
            print(f"  Table '{new_name}' missing fields {sorted(missing)} — deleting and recreating")
            flc.manager.delete_from_definition({"tables": [{"name": new_name}]})
            flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
            existing = None
        else:
            print(f"  Table exists: {new_name} (fields: {len(existing.properties.fields)})")
            return existing

    tdef = build_table_def(src_layer, keep_fields, new_name)
    print(f"  Adding table: {new_name} ({len(tdef['fields'])} fields)")
    flc.manager.add_to_definition({"tables": [tdef]})
    flc = FeatureLayerCollection.fromitem(gis.content.get(item_id))
    return next(t for t in flc.tables if t.properties.name == new_name)

print("Ensuring schema...")
tgt_projects = ensure_table(public_item.id, src_projects, PROJECT_FIELDS, "projects")
tgt_tasks    = ensure_table(public_item.id, src_tasks,    TASK_FIELDS,    "tasks")

## Phase 4 — Refresh data (truncate + append)

Truncate-then-append keeps the public service's item ID and URLs stable across refreshes, so once the slideshow points at the URL it never has to change. Appends in 1,000-row chunks to stay under AGO's per-request limits on larger tables.

In [ ]:
def refresh(src_layer, tgt_layer, fields, label):
    keep_set = set(fields) | {"ObjectId"}
    out_fields = ",".join(sorted(keep_set))
    fset = src_layer.query(where="1=1", out_fields=out_fields, return_geometry=False)
    raw = fset.features
    cleaned = [
        {"attributes": {k: v for k, v in f.attributes.items() if k in keep_set and k != "ObjectId"}}
        for f in raw
    ]
    tgt_layer.manager.truncate()
    added = 0
    for i in range(0, len(cleaned), 1000):
        chunk = cleaned[i:i + 1000]
        result = tgt_layer.edit_features(adds=chunk)
        ok = sum(1 for r in result.get("addResults", []) if r.get("success"))
        added += ok
        if ok != len(chunk):
            fails = [r for r in result.get("addResults", []) if not r.get("success")]
            print(f"  ! {label}: only {ok}/{len(chunk)} added in chunk starting at {i}")
            for fr in fails[:3]:
                print(f"    fail sample: {fr}")
    print(f"  {label}: {added}/{len(cleaned)} rows loaded")
    return added

print("Refreshing data...")
n_proj = refresh(src_projects, tgt_projects, PROJECT_FIELDS, "projects")
n_task = refresh(src_tasks,    tgt_tasks,    TASK_FIELDS,    "tasks")

## Phase 5 — Share publicly

Idempotent: only flips sharing if it isn't already public.

In [ ]:
public_item = gis.content.get(public_item.id)  # refresh metadata
if public_item.access != "public":
    print("Sharing with Everyone...")
    public_item.share(everyone=True)
    print("Shared.")
else:
    print("Already shared with Everyone.")

## Phase 6 — Verification & next steps

Confirms row counts on the public copy match source, and prints the URLs you'll paste into `src/agol.js`.

In [ ]:
flc = FeatureLayerCollection.fromitem(gis.content.get(public_item.id))
print("Public service:")
print(f"  item id: {public_item.id}")
print(f"  access : {public_item.access}")
print()
print("Per-table URLs (paste into ARCGIS_CONFIG in src/agol.js):")
for t in flc.tables:
    count = t.query(where="1=1", return_count_only=True)
    print(f"  {t.properties.name:10s}  rows={count}  url={t.url}")